# Julien's TF pipeline on Colab GPU

Tourne la pipeline (InsightFace + 3DDFA-V2 + BiSeNet) sur le val (15k) et le test (30k) en utilisant un GPU T4.

**ETA approximatif sur T4** : val ~15 min, test ~30 min.

**Pré-requis** : runtime GPU activé (`Exécution → Modifier le type d'exécution → T4`). Dossier `data-challenge-42` sur ton Drive contenant `crops.zip` + `occlusion_datasets/`.

## 1. Setup (~5 min)

In [ ]:
import os, torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')
assert torch.cuda.is_available(), 'No GPU detected. Active T4 dans Runtime settings.'

In [ ]:
REPO_URL = 'https://github.com/StephaneHo/data-challenge-42.git'
REPO_NAME = 'data-challenge-42'
BRANCH = 'feat/zero-shot'

if not os.path.isdir(REPO_NAME):
    !git clone $REPO_URL
%cd $REPO_NAME
!git checkout $BRANCH
!git pull

In [ ]:
# Clone the two extra repos needed by Julien's pipeline
for repo, url in [
    ('3DDFA_V2', 'https://github.com/cleardusk/3DDFA_V2.git'),
    ('face-parsing.PyTorch', 'https://github.com/zllrunning/face-parsing.PyTorch.git'),
]:
    if not os.path.isdir(repo):
        !git clone $url

In [ ]:
# Install python deps
!pip -q install insightface onnxruntime-gpu gdown cython scikit-image imageio imageio-ffmpeg pyyaml

In [ ]:
# Download BiSeNet weights via gdown
import gdown
from pathlib import Path
Path('weights').mkdir(exist_ok=True)
if not Path('weights/79999_iter.pth').exists():
    gdown.download('https://drive.google.com/uc?id=154JgKpzCPW82qINcVieuPH3fZ2e0P812',
                   'weights/79999_iter.pth', quiet=False)
print('BiSeNet weights ready')

## 2. Get the data from Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Adapt to your Drive path
DRIVE_FOLDER = '/content/drive/MyDrive/PRO_et_Mastere/data-challenge-42'
!ls -lh "$DRIVE_FOLDER"

In [ ]:
import os
IMAGE_DIR = '/content/data-challenge-42/crops'
DATA_DIR  = '/content/data-challenge-42/occlusion_datasets'

# crops
if not os.path.isdir(IMAGE_DIR):
    !cp "$DRIVE_FOLDER/crops.zip" /content/crops.zip
    !unzip -q -o /content/crops.zip -d /content/data-challenge-42/
    # Original zip top-level is 'Crop_224_5fp_100K' — rename to 'crops'
    if not os.path.isdir(IMAGE_DIR):
        import glob
        for cand in glob.glob('/content/data-challenge-42/*'):
            if os.path.isdir(cand) and any(os.path.isdir(os.path.join(cand, f'database{i}')) for i in (1, 2, 3)):
                os.rename(cand, IMAGE_DIR)
                break

# occlusion_datasets (CSVs only)
if not os.path.isdir(DATA_DIR):
    os.makedirs(DATA_DIR, exist_ok=True)
    for f in ('train.csv', 'test_students.csv'):
        src = f'{DRIVE_FOLDER}/occlusion_datasets/{f}'
        if os.path.exists(src):
            !cp "$src" "$DATA_DIR/"
        else:
            # The Drive uploaded them as gsheet — fallback: unzip if a zip exists, or upload manually
            print(f'WARNING: {f} not found at {src}. Upload manually into the Colab file pane under {DATA_DIR}/')

print('train.csv:', os.path.exists(f'{DATA_DIR}/train.csv'))
print('crops/ subfolders:', os.listdir(IMAGE_DIR)[:5] if os.path.isdir(IMAGE_DIR) else 'MISSING')

## 3. Patch the run script to use CUDA

In [ ]:
# Force GPU usage in InsightFace (defaults to CPUExecutionProvider in the script)
!sed -i 's/CPUExecutionProvider/CUDAExecutionProvider/g' scripts/zero_shot/run_julien_pipeline.py
# 3DDFA-V2's TDDFA has gpu_mode=False by default; override via cfg keyword
!grep -n 'gpu_mode' 3DDFA_V2/TDDFA.py | head

In [ ]:
# Inject gpu_mode=True into our script's TDDFA call
import fileinput
path = 'scripts/zero_shot/run_julien_pipeline.py'
with open(path) as f:
    code = f.read()
code = code.replace('tddfa = TDDFA(**cfg)', 'cfg["gpu_mode"] = True\n    tddfa = TDDFA(**cfg)')
with open(path, 'w') as f:
    f.write(code)
print('patched')

## 4. Run on val (~10-15 min on T4)

In [ ]:
!python scripts/zero_shot/run_julien_pipeline.py \
  --data-dir $DATA_DIR \
  --image-dir $IMAGE_DIR \
  --source val \
  --out eval/val_julien_baseline.csv

## 5. Run on test (~30 min on T4)

In [ ]:
!python scripts/zero_shot/run_julien_pipeline.py \
  --data-dir $DATA_DIR \
  --image-dir $IMAGE_DIR \
  --source test \
  --out eval/test_julien_baseline.csv

## 6. Backup to Drive + download

In [ ]:
# Save copies to Drive so they survive the Colab session
!cp eval/val_julien_baseline.csv "$DRIVE_FOLDER/val_julien_baseline.csv"
!cp eval/test_julien_baseline.csv "$DRIVE_FOLDER/test_julien_baseline.csv"
print('saved to Drive')

# Also propose direct download
from google.colab import files
files.download('eval/val_julien_baseline.csv')
files.download('eval/test_julien_baseline.csv')